In [ ]:
%%html
<script src="https://unpkg.com/virtual-webgl@1.0.6/src/virtual-webgl.js"></script>

In [ ]:
from IPython.display import HTML


HTML("""
<style>
.plotly-graph-div {
    margin-left: auto !important;
    margin-right: auto !important;
}
</style>
""")

In [ ]:
import os


for var_name, var_value in {
    "PLOTLY_RENDERER": "notebook",
    "XLA_PYTHON_CLIENT_ALLOCATOR": "platform",
    "XLA_PYTHON_CLIENT_PREALLOCATE": "false",
}.items():
    os.environ[var_name] = var_value


os.makedirs("figs", exist_ok=True)

In [ ]:
import glasbey
import h5py
import numpy as np
from plotly import graph_objects as go
from plotly.subplots import make_subplots

from gwkokab.analysis.core.utils import read_attrs_from_hdf5, read_from_hdf5
from gwkokab.analysis.utils.literals import (
    INFERENCE_OUTPUT_FILENAME,
    SAMPLES_GROUP_NAME,
)

In [ ]:
def n_chains_from_hdf5(sampler_name: str, filename: str) -> int:
    if sampler_name == "numpyro":
        with h5py.File(filename, "r") as f:
            return len(f["/chains"].keys())
    return int(read_attrs_from_hdf5(filename, "sampler_cfg")["n_chains"])


def variable_indexes(filename: str) -> dict[str, int]:
    return dict(
        map(
            lambda x: (x[0], int(x[1])),
            read_attrs_from_hdf5(filename, "variables_index").items(),
        )
    )


def n_dims(variable_indexes: dict[str, int]) -> int:
    return max(variable_indexes.values()) + 1


def labels(variable_indexes: dict[str, int]) -> list[str]:
    _n_dims = n_dims(variable_indexes)
    labels = [None] * _n_dims
    for k, v in variable_indexes.items():
        if labels[v] is None:
            labels[v] = k
    assert all(label is not None for label in labels), (
        "Not all dimensions have a label."
    )
    return labels

In [ ]:
inference_data_file = INFERENCE_OUTPUT_FILENAME

In [ ]:
SAMPLER_NAME = read_attrs_from_hdf5(inference_data_file, "sampler_cfg")["sampler_name"]
N_CHAINS = n_chains_from_hdf5(SAMPLER_NAME, inference_data_file)
VARIABLES_INDEX = variable_indexes(inference_data_file)
N_DIMS = n_dims(VARIABLES_INDEX)
LABELS = labels(VARIABLES_INDEX)
SAMPLES = read_from_hdf5(inference_data_file, SAMPLES_GROUP_NAME)


N_SAMPLES, _ = SAMPLES.shape

In [ ]:
CHAINS = np.stack(np.array_split(SAMPLES, N_CHAINS, axis=0), axis=1)

In [ ]:
grid_style = dict(
    showgrid=True,
    gridcolor="rgba(128, 128, 128, 0.5)",
    griddash="dot",
    mirror="all",
    ticks="inside",
    showline=True,
    linewidth=1,
    linecolor="black",
)

In [ ]:
colors_n_chains = glasbey.create_palette(palette_size=N_CHAINS, colorblind_safe=True)
colors_n_dims = glasbey.create_palette(palette_size=N_DIMS, colorblind_safe=True)

In [ ]:
assert len(colors_n_chains) >= N_CHAINS, "Not enough colors for the number of chains."
assert len(colors_n_dims) >= N_DIMS, "Not enough colors for the number of dimensions."

# Trace plot of MCMC samples

This section creates a trace plot to visualize the sampling process of the MCMC algorithm. The trace plot shows the values of each parameter across iterations for all chains, allowing us to assess convergence and mixing of the chains. A well-mixed and converged chain should show stable traces without trends or drifts over iterations.

In [ ]:
n_samples_per_chain, n_chains, n_dims = CHAINS.shape


vertical_spacing = 0.005

fig = make_subplots(
    rows=n_dims,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=vertical_spacing,
)

for i in range(n_dims):
    row = i + 1
    data = CHAINS[..., i]

    for c in range(n_chains):
        show_legend = i == 0

        fig.add_trace(
            go.Scatter(
                y=data[:, c],
                mode="lines",
                line=dict(color=colors_n_chains[c], width=1.5),
                name=f"Chain {c}",
                legendgroup=f"chain_{c}",
                showlegend=show_legend,
            ),
            row=row,
            col=1,
        )


height = max(250, n_dims * 180)

fig.update_layout(
    height=height,
    plot_bgcolor="white",
    margin=dict(l=80, r=60, t=40, b=60),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)


for i in range(n_dims):
    row = i + 1

    fig.update_yaxes(title_text=LABELS[i], **grid_style, row=row, col=1)
    fig.update_xaxes(range=[0, n_samples_per_chain], **grid_style, row=row, col=1)


fig.update_xaxes(title_text="Iteration", row=n_dims, col=1)

fig.write_html("figs/trace_plots.html", include_plotlyjs="cdn", full_html=True)

fig.show()

# Acceptance Rates

<div style="border: 1px solid #00f; background-color: #eef; padding: 10px;">
    <strong>Note:</strong> If a sampler other than flowMC is used, no plots will be generated in this section.
</div>

In [ ]:
if SAMPLER_NAME == "flowMC":
    with h5py.File(inference_data_file, "r") as f:
        global_acc_train = read_from_hdf5(f, "/acceptances/global/train")
        global_acc_prod = read_from_hdf5(f, "/acceptances/global/prod")
        local_acc_train = read_from_hdf5(f, "/acceptances/local/train")
        local_acc_prod = read_from_hdf5(f, "/acceptances/local/prod")

    color_global, color_local = glasbey.create_palette(
        palette_size=2, colorblind_safe=True
    )

    max_len = max(
        len(global_acc_train),
        len(global_acc_prod),
        len(local_acc_train),
        len(local_acc_prod),
    )

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05)

    fig.add_trace(
        go.Scatter(
            y=global_acc_train,
            name="Global Acceptance",
            line=dict(color=color_global),
            legendgroup="global",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            y=local_acc_train,
            name="Local Acceptance",
            line=dict(color=color_local),
            legendgroup="local",
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            y=global_acc_prod,
            name="Global Acceptance",
            line=dict(color=color_global),
            legendgroup="global",
            showlegend=False,  # Avoids duplicate legend items
        ),
        row=2,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            y=local_acc_prod,
            name="Local Acceptance",
            line=dict(color=color_local),
            legendgroup="local",
            showlegend=False,
        ),
        row=2,
        col=1,
    )

    fig.update_layout(
        height=500,
        plot_bgcolor="white",
        margin=dict(l=60, r=50, t=40, b=50),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    )

    fig.update_yaxes(
        title_text="Training Phase", range=[0, 1], **grid_style, row=1, col=1
    )
    fig.update_yaxes(
        title_text="Production Phase",
        range=[0, 1],
        **grid_style,
        row=2,
        col=1,
    )

    fig.update_xaxes(range=[0, max_len], **grid_style, row=1, col=1)
    fig.update_xaxes(
        title_text="Iteration", range=[0, max_len], **grid_style, row=2, col=1
    )

    fig.write_html("figs/acceptance_rates.html", include_plotlyjs="cdn", full_html=True)

    fig.show()

# Normalizing Flow Training Loss

<div style="border: 1px solid #00f; background-color: #eef; padding: 10px;">
    <strong>Note:</strong> If a sampler other than flowMC is used, no plots will be generated in this section.
</div>

In [ ]:
if SAMPLER_NAME == "flowMC":
    loss = read_from_hdf5(inference_data_file, "loss")
    trace = go.Scatter(
        y=loss,
        mode="lines",
        line=dict(width=2),
        showlegend=False,
        hoverinfo="y",
    )

    fig = go.Figure(data=[trace])

    fig.update_layout(
        width=800,
        height=800,
        margin=dict(l=50, r=50, t=50, b=50),
        plot_bgcolor="white",
    )

    fig.update_xaxes(title="Iteration", **grid_style)
    fig.update_yaxes(title="Loss", **grid_style)

    fig.write_html("figs/loss_curve.html", include_plotlyjs="cdn", full_html=True)

    fig.show()

In [ ]:
def auxiliary_chains_plot(datapath: str, output_filename: str) -> None:
    try:
        with h5py.File(inference_data_file, "r") as f:
            chains = np.stack(
                [read_from_hdf5(f, datapath.format(i=i)) for i in range(N_CHAINS)],
                axis=1,
            )
    except Exception:
        return

    fig = make_subplots(
        rows=n_dims,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=vertical_spacing,
    )

    _n_samples = chains.shape[0]

    for i in range(n_dims):
        row = i + 1
        data = chains[..., i]

        for c in range(N_CHAINS):
            show_legend = i == 0

            fig.add_trace(
                go.Scatter(
                    y=data[:, c],
                    mode="lines",
                    line=dict(color=colors_n_chains[c], width=1.5),
                    name=f"Chain {c}",
                    legendgroup=f"chain_{c}",
                    showlegend=show_legend,
                ),
                row=row,
                col=1,
            )

    height = max(250, n_dims * 180)

    fig.update_layout(
        height=height,
        plot_bgcolor="white",
        margin=dict(l=80, r=60, t=40, b=60),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    )

    for i in range(n_dims):
        row = i + 1

        fig.update_yaxes(title_text=LABELS[i], **grid_style, row=row, col=1)
        fig.update_xaxes(range=[0, _n_samples], **grid_style, row=row, col=1)

    fig.update_xaxes(title_text="Iteration", row=n_dims, col=1)

    fig.write_html(output_filename, include_plotlyjs="cdn", full_html=True)

    fig.show()

# Training Chains

<div style="border: 1px solid #00f; background-color: #eef; padding: 10px;">
    <strong>Note:</strong> If a sampler other than flowMC is used, no plots will be generated in this section.
</div>

In [ ]:
if SAMPLER_NAME == "flowMC":
    auxiliary_chains_plot(
        datapath="/chains/train/chain_{i}/positions",
        output_filename="figs/training_trace_plots.html",
    )

# Production Chains

<div style="border: 1px solid #00f; background-color: #eef; padding: 10px;">
    <strong>Note:</strong> If a sampler other than flowMC is used, no plots will be generated in this section.
</div>

In [ ]:
if SAMPLER_NAME == "flowMC":
    auxiliary_chains_plot(
        datapath="/chains/prod/chain_{i}/positions",
        output_filename="figs/production_trace_plots.html",
    )

In [ ]:
from numpyro.diagnostics import (
    autocorrelation,
    autocovariance,
    effective_sample_size,
    split_gelman_rubin,
)


swapped_chains = np.swapaxes(CHAINS, 0, 1)

ess_per_dim = effective_sample_size(swapped_chains)
autocorr_per_dim = autocorrelation(swapped_chains, axis=1)
autocov_per_dim = autocovariance(swapped_chains, axis=1)
rhat_per_dim = split_gelman_rubin(swapped_chains)

## Effective Sample Size


MCMC samples are inherently dependent because each sample is drawn based on the previous one. The Effective Sample Size (ESS) estimates how many independent samples contain the same amount of information as your autocorrelated MCMC chain.

It is computed using the total number of samples across all chains ($N$) and the sum of the autocorrelation estimates at various lags ($\rho_t$):

$$\mathrm{ESS} = \frac{N}{1 + 2 \sum_{t=1}^{\infty} \rho_t}$$

In practice, NumPyro truncates the infinite sum when the autocorrelation noise dominates the signal (Geyer's initial monotone sequence criterion).

## Interpretation

- **Higher is Better:** ESS tells you the "true" size of your dataset for computing posterior statistics (like means, medians, or credible intervals).
- **Rule of Thumb:** A total ESS of $> 400$ is generally acceptable for stable mean and variance estimates, though $> 1000$ is preferred for accurate tail/quantile estimates (like 95% intervals).
- **ESS vs. Actual Samples:** If your total actual samples equal 4,000, but your ESS is 400, it means your highly correlated chain only carries the statistical weight of 400 independent draws.

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=LABELS,
        y=ess_per_dim,
        marker_color=colors_n_dims,
        text=[f"{ess:.3f}" for ess in ess_per_dim],
        textposition="outside",
    )
)
fig.update_layout(
    yaxis_title="Effective Sample Size",
    plot_bgcolor="white",
    margin=dict(l=60, r=50, t=40, b=50),
    xaxis_tickangle=-45,
)
fig.update_xaxes(**grid_style)
fig.update_yaxes(
    type="log",
    # range=[np.log10(np.min(ess_per_dim)) - 0.2, np.log10(np.max(ess_per_dim)) + 0.2],
    **grid_style,
)
fig.write_html(
    "figs/effective_sample_size.html", include_plotlyjs="cdn", full_html=True
)
fig.show()

# Split Gelman-Rubin Diagnostic

The Split Gelman-Rubin diagnostic evaluates MCMC convergence by comparing the variance between multiple independent chains to the variance within those same chains.

NumPyro typically uses the split-$\hat{R}$ method. Each chain is split in half (to detect non-stationarity within a single chain), and then the following are calculated:

- $W$ (Within-chain variance): The average variance of each individual split chain.
- $B$ (Between-chain variance): The variance of the means of the split chains, scaled by the chain length $N$.
- Marginal Posterior Variance: A weighted average of $W$ and $B$, calculated as:
    $$\mathrm{Var}^+(\theta) = \frac{N-1}{N}W + \frac{1}{N}B$$

The diagnostic value is then calculated as the square root of the ratio of this marginal variance to the within-chain variance:

$$\hat{R} = \sqrt{\frac{\text{Var}^+(\theta)}{W}}$$

## Interpretation

- $\hat{R} \approx 1.0$ (Ideal): Indicates that the between-chain variance is identical to the within-chain variance. The chains have mixed well and are likely tracking the same target distribution.
- $\hat{R} > 1.01$ or $1.05$ (Warning): If $\hat{R}$ exceeds these thresholds, it signifies that the chains have not converged. They might be exploring different local modes or haven't run long enough to forget their initial starting positions. Do not use these samples for inference.

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=LABELS,
        y=rhat_per_dim,
        marker_color=colors_n_dims,
        text=[f"{rhat:.3f}" for rhat in rhat_per_dim],
        textposition="outside",
    )
)
fig.add_hline(
    y=1.0,
    line_dash="dash",
    line_color="red",
    annotation_text="R-hat = 1.0 (Ideal)",
    annotation_position="top left",
)
fig.update_layout(
    yaxis_title="Gelman-Rubin Diagnostic",
    plot_bgcolor="white",
    margin=dict(l=60, r=50, t=40, b=50),
    xaxis_tickangle=-45,
)

fig.update_xaxes(**grid_style)


max_r_hat = float(np.max(rhat_per_dim))
min_r_hat = float(np.min(rhat_per_dim))


ymax = max_r_hat + 0.25 * (max_r_hat - int(max_r_hat))
ymin = min(1.0, min_r_hat - 0.25 * abs(min_r_hat - 1.0))

fig.update_yaxes(range=[ymin, ymax], **grid_style)

fig.write_html(
    "figs/gelman_rubin_diagnostic.html", include_plotlyjs="cdn", full_html=True
)
fig.show()

## Autocorrelation


Autocorrelation measures the linear correlation of a chain with a time-lagged version of itself. For a given lag $k$, it calculates how predictive sample $\theta_t$ is of sample $\theta_{t+k}$.

$$\rho_k = \frac{\mathrm{Cov}(\theta_t, \theta_{t+k})}{\mathrm{Var}(\theta_t)}$$

NumPyro outputs this as an array mapping out the correlation coefficients across a range of lags (usually from lag 0 up to the max window size).

## Interpretation

- **Lag 0:** Always equals $1.0$ (a sample is perfectly correlated with itself).
- **Rapid Decay to 0 (Ideal):** In a healthy, fast-mixing chain (like NUTS/HMC), the autocorrelation should drop steeply toward $0$ as the lag increases. This implies successive samples quickly become independent.
- **Slow Decay / High Autocorrelation:** If autocorrelation remains high at large lags (e.g., $\rho_{20} > 0.5$), it means the sampler is taking tiny, sluggish steps through the parameter space. This causes a low ESS and requires running much longer chains.

In [ ]:
fig = make_subplots(
    rows=n_dims,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=vertical_spacing,
)

for i in range(n_dims):
    row = i + 1
    data = autocorr_per_dim[..., i]

    for c in range(n_chains):
        show_legend = i == 0

        fig.add_trace(
            go.Scatter(
                y=data[c],
                mode="lines",
                line=dict(color=colors_n_chains[c], width=1.5),
                name=f"Chain {c}",
                legendgroup=f"chain_{c}",
                showlegend=show_legend,
            ),
            row=row,
            col=1,
        )


height = max(250, n_dims * 180)

fig.update_layout(
    height=height,
    plot_bgcolor="white",
    margin=dict(l=80, r=60, t=40, b=60),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)


for i in range(n_dims):
    row = i + 1

    fig.update_yaxes(title_text=LABELS[i], **grid_style, row=row, col=1)
    fig.update_xaxes(range=[0, n_samples_per_chain], **grid_style, row=row, col=1)


fig.update_xaxes(title_text="Iteration", row=n_dims, col=1)

fig.write_html(
    "figs/autocorrelation_plots.html", include_plotlyjs="cdn", full_html=True
)

fig.show()

In [ ]:
fig = go.Figure()

for i in range(n_dims):
    data = autocorr_per_dim[..., i].mean(0)

    fig.add_trace(
        go.Scatter(
            y=data,
            mode="lines",
            line=dict(color=colors_n_dims[i], width=2.0),
            name=LABELS[i],
            showlegend=True,
        )
    )

fig.add_hline(
    y=0.0,
    line_dash="dash",
    line_color="red",
    annotation_text="Zero Autocorrelation",
    annotation_position="top left",
)

fig.update_layout(
    width=800,
    height=800,
    plot_bgcolor="white",
    margin=dict(l=80, r=60, t=40, b=60),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)


fig.update_yaxes(title_text="Mean Autocorrelation", **grid_style)
fig.update_xaxes(title_text="Iteration", range=[0, n_samples_per_chain], **grid_style)

fig.write_html(
    "figs/mean_autocorrelation_plot.html", include_plotlyjs="cdn", full_html=True
)

fig.show()

## Autocovariance


Autocovariance is the unscaled foundational metric behind autocorrelation. Instead of dividing by the variance to get a standardized $-1$ to $1$ correlation coefficient, it measures the raw covariance of the chain against its lagged self:

$$\gamma_k = E[(\theta_t - \mu)(\theta_{t+k} - \mu)]$$

Where $\mu$ is the empirical mean of the chain.

## Interpretation

- **Scale-Dependent:** Unlike autocorrelation, autocovariance is tied directly to the scale/units of the parameter being measured.
- **Diagnostic Use:** While humans usually prefer looking at autocorrelation because it is standardized, autocovariance is the raw mathematical engine used to compute both the ESS and the spectral density adjustments required for standard error calculations.

In [ ]:
fig = make_subplots(
    rows=n_dims,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=vertical_spacing,
)

for i in range(n_dims):
    row = i + 1
    data = autocov_per_dim[..., i]

    for c in range(n_chains):
        show_legend = i == 0

        fig.add_trace(
            go.Scatter(
                y=data[c],
                mode="lines",
                line=dict(color=colors_n_chains[c], width=1.5),
                name=f"Chain {c}",
                legendgroup=f"chain_{c}",
                showlegend=show_legend,
            ),
            row=row,
            col=1,
        )


height = max(250, n_dims * 180)

fig.update_layout(
    height=height,
    plot_bgcolor="white",
    margin=dict(l=80, r=60, t=40, b=60),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)


for i in range(n_dims):
    row = i + 1

    fig.update_yaxes(title_text=LABELS[i], **grid_style, row=row, col=1)
    fig.update_xaxes(range=[0, n_samples_per_chain], **grid_style, row=row, col=1)


fig.update_xaxes(title_text="Iteration", row=n_dims, col=1)

fig.write_html("figs/autocovariance_plots.html", include_plotlyjs="cdn", full_html=True)

fig.show()

In [ ]:
fig = go.Figure()

for i in range(n_dims):
    data = autocov_per_dim[..., i].mean(0)

    fig.add_trace(
        go.Scatter(
            y=data,
            mode="lines",
            line=dict(color=colors_n_dims[i], width=2.0),
            name=LABELS[i],
            showlegend=True,
        )
    )

fig.update_layout(
    width=800,
    height=800,
    plot_bgcolor="white",
    margin=dict(l=80, r=60, t=40, b=60),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)


fig.update_yaxes(title_text="Mean Autocovariance", **grid_style)
fig.update_xaxes(title_text="Iteration", range=[0, n_samples_per_chain], **grid_style)

fig.write_html(
    "figs/mean_autocovariance_plot.html", include_plotlyjs="cdn", full_html=True
)

fig.show()